# Week 4 — Retrieval-Augmented Generation (RAG)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-04-rag-content.html`. This is the last notebook before your Corte 1 project delivery
in Week 5 — the pipeline you build here is a working draft of that project.

**You will practice:**
1. Chunking a document with overlap.
2. Indexing chunks in ChromaDB, and the same vectors in FAISS for comparison.
3. A complete `answer_with_rag` function.
4. A light ADK agent with a `retrieve_context` tool, and the LangChain equivalent.
5. Two open exercises — including running this on your **own** project documents.


In [2]:
%pip install -q --upgrade ollama python-dotenv numpy chromadb faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [1]:
import ollama
import numpy as np

# Configuración del modelo de Ollama y de embeddings
MODEL = "llama3"
EMBED_MODEL = "nomic-embed-text"

def embed(text):
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return response["embedding"]

## 0. Sample corpus

A small self-contained knowledge base about this course, so the notebook runs without any external files. In the
lab, swap this for **your own project documents** (see Exercise 1).

In [3]:


COURSE_DOCS = """
AI Agentic Engineering is a 16-week elective course for Systems Engineering students at Universidad de Santander. It is organized into three graded cuts called cortes. Corte 1 (weeks 1-5) covers LLM fundamentals, context engineering, and RAG. Corte 2 (weeks 6-11) covers agents, multi-agent systems, Google ADK, and LangGraph. Corte 3 (weeks 12-16) covers evaluation, observability, and deployment to production.

Corte 1 is worth 30% of the final grade: 25% for a practical project and 5% for in-class activities such as quizzes, workshops, and labs. The Corte 1 project requires building a conversational assistant that combines context engineering and RAG over a set of documents chosen by the student, using the Gemini API and ChromaDB, delivered as a GitHub repository with a live 10-minute demonstration.

Corte 2 is also worth 30%: 25% for a multi-agent system project and 5% for in-class activities. Students must build a multi-agent system that solves a real problem using Google ADK or LangGraph, integrating RAG capabilities, delivered with a system diagram and a 15-minute live demonstration.

Corte 3 is worth 40% of the grade: 30% for a final integrator project and 10% for in-class activities. The final project must combine agents, RAG, an automatic evaluation pipeline, tracing with Langfuse, and a REST API exposed with FastAPI, delivered with a 5-minute demo video and a 20-minute technical presentation.

All labs in this course use free-tier tools: the Gemini API through Google AI Studio, ChromaDB and FAISS for vector storage, and Langfuse's free tier for observability starting in Corte 3. Students are also introduced, at a basic level, to Google Antigravity CLI, a terminal-based coding agent that can scaffold and edit project files.

Class time each week is split into two blocks: two hours of theory with instructor-led demonstrations and class discussion, followed by four hours of hands-on lab. Labs mix guided coding, small-group workshops, individual work, and peer code review. All lab work is submitted through the course GitHub repository, with Moodle used for supporting material and announcements.
"""


## 1. Chunking

In [4]:
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 60) -> list[str]:
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(COURSE_DOCS, chunk_size=400, overlap=60)
print(f"{len(COURSE_DOCS)} chars -> {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:70]}...")

2137 chars -> 7 chunks
[0] AI Agentic Engineering is a 16-week elective course for Systems Engine...
[1] s 12-16) covers evaluation, observability, and deployment to productio...
[2] s chosen by the student, using the Gemini API and ChromaDB, delivered ...
[3]  capabilities, delivered with a system diagram and a 15-minute live de...
[4]  5-minute demo video and a 20-minute technical presentation. All labs ...
[5] d coding agent that can scaffold and edit project files. Class time ea...
[6]  the course GitHub repository, with Moodle used for supporting materia...


## 2. Indexing with ChromaDB

In [5]:
import numpy as np
import ollama

# Modelo ligero dedicado solo a embeddings (evita saturar la RAM)
EMBED_MODEL = "nomic-embed-text"


def embed(text: str) -> list[float]:
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return response["embedding"]

In [9]:
import gc
import chromadb
import numpy as np

# Cliente en memoria (no toca el disco duro, evita el crash de SQLite/Windows)
chroma_client = chromadb.EphemeralClient()

collection = chroma_client.create_collection(name="course_docs")

for i, chunk in enumerate(chunks):
    vector = np.array(embed(chunk), dtype=np.float32).squeeze().tolist()

    collection.add(
        documents=[chunk],
        embeddings=[vector],
        ids=[f"chunk-{i}"],
    )
    gc.collect()

print(
    f"¡Éxito sin crashes! {collection.count()} chunks indexados en memoria."
)

: 

## 3. A complete `answer_with_rag` function

In [8]:
import faiss
import numpy as np
import ollama

# 1. Crear el índice de FAISS con los chunks del documento
vectors = np.array([embed(c) for c in chunks], dtype="float32")
faiss_index = faiss.IndexFlatL2(vectors.shape[1])
faiss_index.add(vectors)

# Modelo de chat de Ollama
LLM_MODEL = "qwen2.5:14b"  # o "deepseek-r1:14b" / "llama3"


# 2. Función RAG
def answer_with_rag(
    question: str, n_results: int = 3, verbose: bool = False
) -> str:
    # Búsqueda por similitud en FAISS
    query_vector = np.array([embed(question)], dtype="float32")
    distances, indices = faiss_index.search(query_vector, k=n_results)

    # Extraer los chunks recuperados
    retrieved_chunks = [chunks[idx] for idx in indices[0]]

    if verbose:
        print("--- retrieved chunks ---")
        for c in retrieved_chunks:
            print("-", c[:80], "...")

    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the answer isn't
in the context, say you don't have that information — do not make anything up.

Context:
{context}

Question: {question}
Answer:"""

    response = ollama.generate(
        model=LLM_MODEL,
        prompt=prompt,
        options={"temperature": 0.1},
    )

    return response["response"]


# 3. Pruebas
print(answer_with_rag("How is Corte 1 graded?", verbose=True))
print("\n" + "=" * 50 + "\n")
print(answer_with_rag("What programming language does the course use?"))

--- retrieved chunks ---
- s 12-16) covers evaluation, observability, and deployment to production. Corte 1 ...
-  capabilities, delivered with a system diagram and a 15-minute live demonstratio ...
- s chosen by the student, using the Gemini API and ChromaDB, delivered as a GitHu ...
Corte 1 is graded with 30% of the final grade, broken down into 25% for a practical project involving the creation of a conversational assistant, and 5% for in-class activities such as quizzes, workshops, and labs.


The given context does not specify the programming language used in the course.


## 4. Light preview: RAG as an ADK tool, and the LangChain equivalent

We're one step away from a real agent (Week 6 onward covers tool use and ReAct properly). For now, notice that
"retrieval" is just a Python function — which is exactly what an ADK **tool** is.

In [9]:
import numpy as np
import ollama

# 1. Define the tool function using FAISS
def retrieve_context(query: str) -> str:
    """Retrieve the most relevant document chunks for a query from FAISS."""
    query_vector = np.array([embed(query)], dtype="float32")
    distances, indices = faiss_index.search(query_vector, k=3)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]
    return "\n\n---\n\n".join(retrieved_chunks)


# 2. Function to execute Ollama Agent with tool calling
def ask_ollama_tool_agent(prompt: str, model: str = "qwen2.5:14b") -> str:
    # System instructions
    system_instruction = (
        "You are a helpful assistant. Use the retrieve_context tool to find relevant chunks "
        "before answering. Answer ONLY using information returned by the tool. If it's not there, say so."
    )

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": prompt},
    ]

    # First call: LLM decides whether to call the tool
    response = ollama.chat(
        model=model,
        messages=messages,
        tools=[retrieve_context],
    )

    # Check if the model invoked the retrieve_context tool
    if response.message.tool_calls:
        for tool in response.message.tool_calls:
            if tool.function.name == "retrieve_context":
                query_arg = tool.function.arguments.get("query", prompt)
                tool_output = retrieve_context(query_arg)

                # Append tool call and tool response to chat history
                messages.append(response.message)
                messages.append(
                    {
                        "role": "tool",
                        "content": tool_output,
                    }
                )

        # Second call: LLM answers using retrieved context
        final_response = ollama.chat(model=model, messages=messages)
        return final_response.message.content

    return response.message.content


# Test ADK tool equivalence with Ollama
print(ask_ollama_tool_agent("How is Corte 2 graded?"))

Corte 2 is worth 30% of the final grade, with 25% allocated for a multi-agent system project and 5% for in-class activities such as quizzes, workshops, and labs. The project requires building a multi-agent system that addresses a real-world problem using Google ADK or LangGraph, incorporating RAG capabilities. Students must deliver their project with a system diagram and provide a 15-minute live demonstration.


In [10]:
from langchain_ollama import ChatOllama

# 1. Initialize Ollama LLM in LangChain
llm = ChatOllama(model="qwen2.5:14b", temperature=0.1)


# 2. LangChain RAG function using FAISS
def answer_with_rag_langchain(question: str, n_results: int = 3) -> str:
    # Retrieve top-k chunks from FAISS
    query_vector = np.array([embed(question)], dtype="float32")
    distances, indices = faiss_index.search(query_vector, k=n_results)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]

    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = (
        "Answer the question using ONLY the context below. If the answer isn't in the "
        "context, say you don't have that information.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )

    return llm.invoke(prompt).content


# Test LangChain equivalence with Ollama
print(
    answer_with_rag_langchain(
        "What tools are used for observability in this course?"
    )
)

The tool used for observability in this course is Langfuse's free tier, starting in Corte 3.


## 5. Exercises

In [12]:
import glob
import os
import faiss
import numpy as np
import ollama

# 1. Ensure data directory exists and has files
data_folder = "./data"
os.makedirs(data_folder, exist_ok=True)

# Create fallback files if data directory is empty
sample_files = glob.glob(os.path.join(data_folder, "*.txt"))
if not sample_files:
    sample_doc = os.path.join(data_folder, "cyber_security_project.txt")
    with open(sample_doc, "w", encoding="utf-8") as f:
        f.write(
            """RAY Cyber-Madurez Core Framework Specification.
The main objective of the RAY Cyber-Madurez Core framework is to evaluate and improve information security maturity in small and medium enterprises (SMEs).
Supported frameworks include ISO 27001, ISO 25010 for software quality, and NIST Cybersecurity Framework.
The MVP architecture contains core modules: Assessment Engine, Risk Management Core, and Security Reporting Dashboard.
Integration test cases cover functional API endpoints, authentication security, and ISO 25010 compliance validation."""
        )

# Load all plain-text files from the data/ folder
document_files = glob.glob(os.path.join(data_folder, "*.txt"))

all_chunks = []
chunk_metadata = []

# Simple character-based chunking with overlap
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

for file_path in document_files:
    file_name = os.path.basename(file_path)
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunk_text = text[start:end]
        all_chunks.append(chunk_text)
        chunk_metadata.append({"source": file_name, "start": start})
        start += CHUNK_SIZE - CHUNK_OVERLAP

print(
    f"Loaded {len(document_files)} document(s) and created {len(all_chunks)} chunk(s)."
)

# 2. Embed chunks and create FAISS index
chunk_vectors = np.array([embed(c) for c in all_chunks], dtype="float32")
project_faiss_index = faiss.IndexFlatL2(chunk_vectors.shape[1])
project_faiss_index.add(chunk_vectors)

# 3. Define answer_with_rag for Exercise 1
LLM_MODEL = "qwen2.5:14b"


def answer_project_rag(
    question: str, n_results: int = 3, verbose: bool = False
) -> str:
    query_vector = np.array([embed(question)], dtype="float32")
    distances, indices = project_faiss_index.search(
        query_vector, k=min(n_results, len(all_chunks))
    )

    retrieved_chunks = [all_chunks[idx] for idx in indices[0]]

    if verbose:
        print("--- RETRIEVED CHUNKS ---")
        for idx in indices[0]:
            print(
                f"[{chunk_metadata[idx]['source']}] {all_chunks[idx][:80]}..."
            )

    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""You are a helpful assistant for the RAY Cyber-Madurez Core project.
Answer the user question using ONLY the provided context below.
If the answer is not in the context, explicitly state that you do not have that information.

Context:
{context}

Question: {question}
Answer:"""

    response = ollama.generate(
        model=LLM_MODEL, prompt=prompt, options={"temperature": 0.1}
    )
    return response["response"]


# 4. Test with 5 real questions
questions = [
    "What is the main objective of the RAY Cyber-Madurez Core framework?",
    "Which ISO standards or cybersecurity frameworks are supported?",
    "How does the framework evaluate cybersecurity maturity in SMEs?",
    "What core components or modules are included in the MVP architecture?",
    "How are integration test cases structured for the security module?",
]

for i, q in enumerate(questions, 1):
    print(f"=== Question {i}: {q} ===")
    print(answer_project_rag(q, n_results=3, verbose=True))
    print("\n" + "=" * 60 + "\n")

Loaded 1 document(s) and created 3 chunk(s).
=== Question 1: What is the main objective of the RAY Cyber-Madurez Core framework? ===
--- RETRIEVED CHUNKS ---
[cyber_security_project.txt] RAY Cyber-Madurez Core Framework Specification.
The main objective of the RAY Cy...
[cyber_security_project.txt] 010 for software quality, and NIST Cybersecurity Framework.
The MVP architecture...
[cyber_security_project.txt] ecurity, and ISO 25010 compliance validation....
The main objective of the RAY Cyber-Madurez Core framework is to evaluate and improve information security maturity in small and medium enterprises (SMEs).


=== Question 2: Which ISO standards or cybersecurity frameworks are supported? ===
--- RETRIEVED CHUNKS ---
[cyber_security_project.txt] RAY Cyber-Madurez Core Framework Specification.
The main objective of the RAY Cy...
[cyber_security_project.txt] 010 for software quality, and NIST Cybersecurity Framework.
The MVP architecture...
[cyber_security_project.txt] ecurity, and ISO 

In [13]:
import numpy as np
import ollama
import faiss

def answer_with_metadata_filter(
    question: str, target_source: str, n_results: int = 3
) -> str:
    """Queries the index but restricts retrieval strictly to chunks matching target_source."""
    
    # 1. Filter valid chunk indices matching the target source file
    valid_indices = [
        i for i, meta in enumerate(chunk_metadata)
        if meta["source"] == target_source
    ]

    # Safeguard: Check if target document exists in metadata
    if not valid_indices:
        return f"Error: No chunks found for source document '{target_source}'."

    # 2. Extract sub-matrix of vectors matching target_source
    filtered_vectors = chunk_vectors[valid_indices]

    # Safeguard: Verify sub-matrix shape before creating FAISS index
    if filtered_vectors.ndim < 2 or filtered_vectors.shape[0] == 0:
        return f"Error: Vector matrix for '{target_source}' is empty or invalid."

    # 3. Build a temporary FAISS index for the filtered subset
    temp_index = faiss.IndexFlatL2(filtered_vectors.shape[1])
    temp_index.add(filtered_vectors)

    # Search in filtered subset (capping k to available filtered chunks)
    k_actual = min(n_results, len(valid_indices))
    query_vector = np.array([embed(question)], dtype="float32")
    distances, sub_indices = temp_index.search(query_vector, k=k_actual)

    # 4. Map sub-indices back to global all_chunks
    retrieved_chunks = []
    print(f"--- FILTERED RETRIEVAL (source: '{target_source}') ---")
    for sub_idx in sub_indices[0]:
        global_idx = valid_indices[sub_idx]
        retrieved_chunks.append(all_chunks[global_idx])
        print(f"- [{chunk_metadata[global_idx]['source']}] {all_chunks[global_idx][:80]}...")

    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""You are a helpful assistant for the RAY Cyber-Madurez Core project.
Answer the user question strictly using ONLY the context below.
If the answer is not in the context, say you don't have that information.

Context:
{context}

Question: {question}
Answer:"""

    response = ollama.generate(
        model=LLM_MODEL, prompt=prompt, options={"temperature": 0.1}
    )
    return response["response"]


# Example Test: Dynamically target the first available document source
if chunk_metadata:
    target_file = chunk_metadata[0]["source"]
    test_question = "What are the primary functional requirements or modules defined in this document?"

    print(
        answer_with_metadata_filter(
            test_question, target_source=target_file, n_results=2
        )
    )
else:
    print("No chunks available. Please run Exercise 1 first to index your documents.")

--- FILTERED RETRIEVAL (source: 'cyber_security_project.txt') ---
- [cyber_security_project.txt] 010 for software quality, and NIST Cybersecurity Framework.
The MVP architecture...
- [cyber_security_project.txt] RAY Cyber-Madurez Core Framework Specification.
The main objective of the RAY Cy...
The primary functional requirements or modules defined in this document are the core components of the MVP architecture, which include the Assessment Engine, Risk Management Core, and Security Reporting Dashboard.


## Corte 1 wrap-up

You now have every piece needed for your Week 5 project: a system prompt (Week 2), context/history handling
(Week 3), and a working RAG pipeline (this week). See `week-04-rag-activities.html` for the full delivery
checklist and grading rubric.

## Looking ahead

Corte 2 starts in Week 6 with agent fundamentals — tool use, function calling, and the ReAct pattern — building
directly on the `tools=[retrieve_context]` preview above.